# Reddit Scraper
Perform scraping from subreddits related to football using Python Reddit API Wrapper (PRAW)

## Subreddits used
### Main subreddits
* soccer
* premierleague
* football
* theother14
### Big clubs
* reddevils
* LiverpoolFC
* gunners
* chelseafc
* MCFC       
* coys       
### Others
* FantasyPL
* Championship
### Diversity subreddits
* realmadrid
* Barca
* fcbayern
* psg
* ACMilan

## Process
Take the latest 300 posts and their comments from each other subreddits above, skipping moderator comments.
Afterwards, export this information as a csv to be used for data cleaning and subsequently, model training.

### post information taken
* author
* author_flair
* title
* content
* post_id
* date
* upvotes
* upvote_ratio
* num_comments
* comments
* subreddit

In [ ]:
import praw as pp
from praw.models import MoreComments as mc 
import pandas as pd

In [ ]:
reddit = pp.Reddit( client_id = "s0Epgs8jsHkNKzsFGCnw-w",
                    client_secret = None,
                    redirect_uri="http://localhost:8080",
                    user_agent = "python:school_project:v1.0.0 (by u/4021-30)")

subreddits = [
    # main subreddits
    "soccer",
    "premierleague",
    "football",
    "theother14",

    # big clubs
    "reddevils",    # man u
    "LiverpoolFC",
    "gunners",      # arsenal
    "chelseafc",
    "MCFC",         # man city
    "coys",         # spurs
    
    # others
    "FantasyPL",
    "Championship",
    
    # diversity subreddits
    "realmadrid",
    "Barca",
    "fcbayern",
    "psg",
    "ACMilan",
]

Use praw to go through the 300 latest posts and get their comments, skipping moderator comments.

In [ ]:
# send the collected data into a csv, can use for classification in the classification.ipynb file
dfRows = []
commentCount = 0
postCount = 0
for sub in subreddits:
    postCount = 0
    for submission in reddit.subreddit(sub).new(limit=300):
        postCount += 1
        commentList = []
        print("r/{} post {}, total comments: {}".format(sub, postCount, commentCount))
        for c in submission.comments:
            commentCount += 1
            if (type(c) == mc):
                continue

            if (c.distinguished == "moderator"):
                print("skipping mod comment")
                commentCount -= 1
                continue

            commentList.append(c.body)

        tempRow = {
            "author"        : str(submission.author),
            "author_flair"  : submission.author_flair_text,
            "title"         : submission.title,
            "content"       : submission.selftext,
            "post_id"       : submission.id,
            "date"          : submission.created_utc,
            "upvotes"       : submission.score,
            "upvote_ratio"  : submission.upvote_ratio,
            "num_comments"  : submission.num_comments,
            "comments"      : commentList,
            "subreddit"     : submission.subreddit.display_name
        }
        dfRows.append(tempRow)

df = pd.DataFrame(dfRows)
print("found {} rows".format(len(df)))


After collecting all of the information from Reddit, convert it to CSV to be used later.

In [ ]:
df.to_csv("reddit-scrape-data/data.csv", index = False)